# Time Series Forecasting Using Python

This notebook analyzes monthly revenue data, performs trend and seasonality analysis, builds forecasting models, and evaluates prediction performance.

**Tools used:** Python, Pandas, Matplotlib, Statsmodels, SARIMA, and optional Prophet.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams['figure.figsize'] = (10, 5)

## 1. Load Dataset

The dataset contains monthly revenue values. The date column is converted to datetime format and set as the index for time series analysis.

In [ ]:
df = pd.read_csv('data/dataset.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date').asfreq('MS')
df.head()

## 2. Basic Data Check

In [ ]:
print(df.info())
print('Missing values:')
print(df.isna().sum())
print(df.describe())

## 3. Trend Visualization

In [ ]:
plt.plot(df.index, df['revenue'], marker='o')
plt.title('Monthly Revenue Trend')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Seasonal Decomposition

This step separates the time series into trend, seasonal, and residual components.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(df['revenue'], model='additive', period=12)
decomposition.plot()
plt.tight_layout()
plt.show()

## 5. Train-Test Split

The last 12 months are used as test data to compare model predictions with actual values.

In [ ]:
train = df.iloc[:-12]
test = df.iloc[-12:]

print('Training rows:', len(train))
print('Testing rows:', len(test))

## 6. SARIMA Forecasting Model

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(
    train['revenue'],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_result = model.fit(disp=False)
sarima_forecast = sarima_result.forecast(steps=len(test))
sarima_forecast.head()

## 7. Forecast Visualization

In [ ]:
plt.plot(train.index, train['revenue'], label='Training Data')
plt.plot(test.index, test['revenue'], label='Actual Test Data', marker='o')
plt.plot(test.index, sarima_forecast, label='SARIMA Forecast', marker='x')
plt.title('SARIMA Forecast vs Actual Revenue')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Model Evaluation

In [ ]:
mae = mean_absolute_error(test['revenue'], sarima_forecast)
rmse = np.sqrt(mean_squared_error(test['revenue'], sarima_forecast))
mape = np.mean(np.abs((test['revenue'] - sarima_forecast) / test['revenue'])) * 100

metrics = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'MAPE (%)'],
    'Value': [mae, rmse, mape]
})
metrics

In [ ]:
plt.bar(metrics['Metric'], metrics['Value'])
plt.title('Forecast Model Evaluation')
plt.ylabel('Value')
plt.tight_layout()
plt.show()

## 9. Optional Prophet Model

Run this section only if Prophet is installed in your environment. If Prophet is not installed, the SARIMA section above is enough for the main forecasting workflow.

In [ ]:
# Optional Prophet workflow
# from prophet import Prophet
#
# prophet_df = df.reset_index().rename(columns={'date': 'ds', 'revenue': 'y'})
# prophet_train = prophet_df.iloc[:-12]
# prophet_test = prophet_df.iloc[-12:]
#
# prophet_model = Prophet(yearly_seasonality=True)
# prophet_model.fit(prophet_train)
# future = prophet_model.make_future_dataframe(periods=12, freq='MS')
# prophet_forecast = prophet_model.predict(future)
# prophet_model.plot(prophet_forecast)
# plt.title('Prophet Forecast')
# plt.show()

## Conclusion

The project shows how monthly time series data can be cleaned, visualized, decomposed, modeled, forecasted, and evaluated. SARIMA is useful for statistical forecasting when seasonality is present, while Prophet can be added as a flexible alternative for trend and seasonality-based forecasting.